#### Data preprocessing and Splitting


In [ ]:
import os
import shutil
import random
from PIL import Image, ImageEnhance
from torchvision import transforms

# Define paths
original_root = "D:\\College\\Sem 6\\Biometric Security\\deepfake_detection_project\\data\\original_sequences\\actors\\c23\\images_1"
manipulated_root = "D:\\College\\Sem 6\\Biometric Security\\deepfake_detection_project\\data\\manipulated_sequences\\DeepFakeDetection\\c23\\images_1"
output_dir = "D:\\College\\Sem 6\\Biometric Security\\deepfake_detection_project\\data\\dataset_temp"  # Base output directory

# Ratios for splitting data
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# Desired image size for the CNN (224x224 as in original setup)
image_size = (224, 224)

# Create the train, validation, and test directories
splits = ['training_set', 'validation_set', 'testing_set']
for split in splits:
    os.makedirs(os.path.join(output_dir, split, 'original'), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, 'manipulated'), exist_ok=True)

# Get all folders
all_folders = os.listdir(manipulated_root)
random.shuffle(all_folders)

# Calculate split sizes
num_folders = len(all_folders)
train_split = int(num_folders * train_ratio)
val_split = int(num_folders * val_ratio)

# Divide the folders into splits
train_folders = all_folders[:train_split]
val_folders = all_folders[train_split:train_split + val_split]
test_folders = all_folders[train_split + val_split:]

# Define preprocessing and augmentation transformations
preprocess_transform = transforms.Compose([
    transforms.Resize(image_size),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Augment brightness and contrast
    # transforms.RandomHorizontalFlip(p=0.5),  # Random horizontal flip
    transforms.ToTensor(),  # Convert image to tensor (scales to 0-1 automatically)
])

def process_and_save_image(image_path, output_path):
    # Apply preprocessing to a single image and save it to the specified path.
    
    with Image.open(image_path) as img:
        # Apply transformations and save image
        img = preprocess_transform(img)
        img = transforms.ToPILImage()(img)  # Convert back to PIL to save as image
        img.save(output_path)

def copy_and_preprocess_images(folders, split):
    # Copy and preprocess images from original and manipulated folders to the split directory.
    
    for folder in folders:
        # Define paths for original and manipulated folders
        original_folder = os.path.join(original_root, folder, 'original')
        manipulated_folder = os.path.join(manipulated_root, folder, 'original')

        # Define output paths
        output_original = os.path.join(output_dir, split, 'original', folder)
        output_manipulated = os.path.join(output_dir, split, 'manipulated', folder)

        # Create directories if they don't exist
        os.makedirs(output_original, exist_ok=True)
        os.makedirs(output_manipulated, exist_ok=True)

        # Process and save images
        if os.path.exists(original_folder):
            for img_name in os.listdir(original_folder):
                img_path = os.path.join(original_folder, img_name)
                output_img_path = os.path.join(output_original, img_name)
                process_and_save_image(img_path, output_img_path)

        if os.path.exists(manipulated_folder):
            for img_name in os.listdir(manipulated_folder):
                img_path = os.path.join(manipulated_folder, img_name)
                output_img_path = os.path.join(output_manipulated, img_name)
                process_and_save_image(img_path, output_img_path)

# Process and save images for each split
copy_and_preprocess_images(train_folders, 'training_set')
copy_and_preprocess_images(val_folders, 'validation_set')
copy_and_preprocess_images(test_folders, 'testing_set')

print("Dataset has been split and preprocessed.")


Dataset has been split and preprocessed.


In [ ]:
import os
import shutil
import random
import pandas as pd
from PIL import Image
import numpy as np
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torchvision.transforms.functional import to_pil_image

# Load and process the CSV file
def load_folder_data(csv_path):
    df = pd.read_csv(csv_path)
    folder_pairs = list(zip(df['manipulated_folder'], df['original_folder']))
    # print(folder_pairs)
    image_counts = list(zip(df['manipulated_image_count'], df['original_image_count']))
    # print(image_counts)
    return folder_pairs, image_counts

# Define paths
original_root = "D:\\College\\Sem 6\\Biometric Security\\deepfake_detection_project\\data\\original_sequences\\actors\\c23\\images_1"
manipulated_root = "D:\\College\\Sem 6\\Biometric Security\\deepfake_detection_project\\data\\manipulated_sequences\\DeepFakeDetection\\c23\\images_1"
output_dir = "D:\\College\\Sem 6\\Biometric Security\\deepfake_detection_project\\data\\dataset_set"  # Base output directory

# Split ratios
train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# Image preprocessing parameters
image_size = (224, 224)
# mean = [0.485, 0.456, 0.406]  # ImageNet mean
# std = [0.229, 0.224, 0.225]   # ImageNet std

# Create preprocessing pipeline
preprocess_transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    # transforms.Normalize(mean=mean, std=std)
])

# Create augmentation pipeline for training
train_transform = transforms.Compose([
    transforms.Resize(image_size),
    # transforms.RandomHorizontalFlip(),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    # transforms.Normalize(mean=mean, std=std)
])

def create_directory_structure():
    """Create the necessary directory structure"""
    splits = ['train', 'val', 'test']
    for split in splits:
        for category in ['original', 'manipulated']:
            os.makedirs(os.path.join(output_dir, split, category), exist_ok=True)

def process_and_save_image(image_path, output_path, transform):
    try:
        # print(image_path)
        with Image.open(image_path) as img:
            # if img.mode != 'RGB':
            #     img = img.convert('RGB')
            
            # Apply preprocessing
            processed_img = transform(img)
            
            # Convert back to PIL image for saving
            processed_img = to_pil_image(processed_img)
            processed_img.save(output_path)
            return True
    except Exception as e:
        print(f"Error processing image {image_path}: {e}")
        return False

def copy_and_preprocess_folder(src_folder, dst_folder, transform):
    """Copy and preprocess all images in a folder"""
    os.makedirs(dst_folder, exist_ok=True)
    success_count = 0
    failed_count = 0

    print(src_folder)
    for img_name in os.listdir(src_folder):
        # print("image names", img_name)
        if img_name.lower().endswith(('.png')):
            src_path = os.path.join(src_folder, img_name)
            dst_path = os.path.join(dst_folder, img_name)
            
            if process_and_save_image(src_path, dst_path, transform):
                success_count += 1
            else:
                failed_count += 1
    
    return success_count, failed_count

def main(csv_path):
    # Load folder information from CSV
    folder_pairs, _ = load_folder_data(csv_path)
    
    # Split folders into train, validation, and test sets
    train_pairs, temp_pairs = train_test_split(folder_pairs, train_size=train_ratio, random_state=42)
    # print(len(train_pairs))
    val_pairs, test_pairs = train_test_split(temp_pairs, train_size=0.5, random_state=42)
    # print(len(test_pairs))
    # print(len(val_pairs))

    
    # Create directory structure
    create_directory_structure()
    
    # Process each split
    splits = {
        'train': (train_pairs, train_transform),
        'val': (val_pairs, preprocess_transform),
        'test': (test_pairs, preprocess_transform)
    }
    
    for split_name, (pairs, transform) in splits.items():
        # print(f"\nProcessing {split_name} split...")
        
        for manipulated_folder, original_folder in pairs:
            # Process manipulated images
            src_manipulated = os.path.join(manipulated_root, manipulated_folder+"\original")
            # print(src_manipulated)
            dst_manipulated = os.path.join(output_dir, split_name, 'manipulated', manipulated_folder)
            # print(dst_manipulated)
            
            # Process original images
            
            src_original = os.path.join(original_root, original_folder+"\original")
            dst_original = os.path.join(output_dir, split_name, 'original', original_folder)
            
            # print(f"Processing folder pair: {manipulated_folder} - {original_folder}")
            
            # Copy and preprocess images
            m_success, m_failed = copy_and_preprocess_folder(src_manipulated, dst_manipulated, transform)
            o_success, o_failed = copy_and_preprocess_folder(src_original, dst_original, transform)
            
            # print(f"Manipulated: {m_success} successful, {m_failed} failed")
            # print(f"Original: {o_success} successful, {o_failed} failed")


if __name__ == "__main__":
    csv_path = "../data/folder_image_counts.csv"  # Path to your CSV file
    main(csv_path)

D:\College\Sem 6\Biometric Security\deepfake_detection_project\data\manipulated_sequences\DeepFakeDetection\c23\images_1\26_25__walking_down_street_outside_angry__PQ41U3IJ\original
Image saved to D:\College\Sem 6\Biometric Security\deepfake_detection_project\data\dataset_set\train\manipulated\26_25__walking_down_street_outside_angry__PQ41U3IJ\0000.png
Image saved to D:\College\Sem 6\Biometric Security\deepfake_detection_project\data\dataset_set\train\manipulated\26_25__walking_down_street_outside_angry__PQ41U3IJ\0001.png
Image saved to D:\College\Sem 6\Biometric Security\deepfake_detection_project\data\dataset_set\train\manipulated\26_25__walking_down_street_outside_angry__PQ41U3IJ\0002.png
Image saved to D:\College\Sem 6\Biometric Security\deepfake_detection_project\data\dataset_set\train\manipulated\26_25__walking_down_street_outside_angry__PQ41U3IJ\0003.png
Image saved to D:\College\Sem 6\Biometric Security\deepfake_detection_project\data\dataset_set\train\manipulated\26_25__walking

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import VGG16_Weights
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def check_gpu():
    print("\n=== GPU Setup ===")
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA version:", torch.version.cuda)
        print("GPU device count:", torch.cuda.device_count())
        print("GPU device name:", torch.cuda.get_device_name(0))
        print("Current GPU device:", torch.cuda.current_device())
    print("================\n")

class DeepFakeDetector(nn.Module):
    def __init__(self, freeze_backbone=True):
        super(DeepFakeDetector, self).__init__()
        self.vgg16 = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        
        if freeze_backbone:
            for param in self.vgg16.features.parameters():
                param.requires_grad = False
        
        self.vgg16.classifier[6] = nn.Linear(4096, 2)
        
    def forward(self, x):
        return self.vgg16(x)

def train_epoch(model, dataloader, criterion, optimizer, device, scheduler=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        if batch_idx % 10 == 0:  # Print every 10 batches
            print(f'Batch [{batch_idx}/{len(dataloader)}] '
                  f'Loss: {loss.item():.4f} '
                  f'Acc: {100.*correct/total:.2f}% '
                  f'GPU Memory: {torch.cuda.memory_allocated()/1024**2:.1f}MB')
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    val_loss = running_loss / len(dataloader)
    val_acc = accuracy_score(all_labels, all_preds) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    
    return val_loss, val_acc, precision, recall, f1

def main():
    # Set for deterministic behavior and performance
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

    # Check GPU setup
    check_gpu()
    
    # Set device and ensure it's using CUDA
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    if device.type == 'cuda':
        print(f'Initial GPU Memory Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB')
        print(f'Initial GPU Memory Reserved: {torch.cuda.memory_reserved(0)/1024**2:.2f} MB')
    
    # Dataset paths
    data_dir = "../data/dataset"
    
    # Hyperparameters
    batch_size = 128  # Increased for GPU
    num_epochs = 20
    learning_rate = 0.001
    
    # Data transforms
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    
    # Load datasets
    try:
        train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
        val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
        test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=transform)
        
        print(f"\nDataset Statistics:")
        print(f"Training images: {len(train_dataset)}")
        print(f"Validation images: {len(val_dataset)}")
        print(f"Test images: {len(test_dataset)}")
        print(f"Classes: {train_dataset.classes}\n")
        
    except Exception as e:
        print(f"Error loading datasets: {e}")
        print(f"Data directory: {os.path.abspath(data_dir)}")
        return
    
    # Create dataloaders with multiple workers for GPU
    train_loader = DataLoader(train_dataset, 
                            batch_size=batch_size, 
                            shuffle=True,
                            num_workers=4,
                            pin_memory=True)
    val_loader = DataLoader(val_dataset, 
                          batch_size=batch_size,
                          num_workers=4,
                          pin_memory=True)
    test_loader = DataLoader(test_dataset, 
                           batch_size=batch_size,
                           num_workers=4,
                           pin_memory=True)
    
    # Initialize model
    model = DeepFakeDetector(freeze_backbone=True)
    print("\nModel Device Check:")
    print("Model device before:", next(model.parameters()).device)
    model = model.to(device)
    print("Model device after:", next(model.parameters()).device, "\n")
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=learning_rate,
        epochs=num_epochs,
        steps_per_epoch=len(train_loader)
    )
    
    # Training history
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': []
    }
    
    # Training loop
    best_val_acc = 0.0
    try:
        for epoch in range(num_epochs):
            print(f"\nEpoch [{epoch+1}/{num_epochs}]")
            
            if device.type == 'cuda':
                print(f'GPU Memory Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB')
                print(f'GPU Memory Reserved: {torch.cuda.memory_reserved(0)/1024**2:.2f} MB')
            
            # Train
            train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device, scheduler)
            
            # Validate
            val_loss, val_acc, precision, recall, f1 = validate(model, val_loader, criterion, device)
            
            # Update history
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            history['val_precision'].append(precision)
            history['val_recall'].append(recall)
            history['val_f1'].append(f1)
            
            print(f"\nEpoch Summary:")
            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
            print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
            print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")
            
            # Save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'best_val_acc': best_val_acc,
                }, 'best_model.pth')
                print(f"Saved new best model with validation accuracy: {val_acc:.2f}%")
        
        # Plot training history
        plt.figure(figsize=(15, 5))
        
        plt.subplot(1, 3, 1)
        plt.plot(history['train_loss'], label='Train Loss')
        plt.plot(history['val_loss'], label='Val Loss')
        plt.title('Loss vs Epoch')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        
        plt.subplot(1, 3, 2)
        plt.plot(history['train_acc'], label='Train Acc')
        plt.plot(history['val_acc'], label='Val Acc')
        plt.title('Accuracy vs Epoch')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy (%)')
        plt.legend()
        
        plt.subplot(1, 3, 3)
        plt.plot(history['val_precision'], label='Precision')
        plt.plot(history['val_recall'], label='Recall')
        plt.plot(history['val_f1'], label='F1')
        plt.title('Metrics vs Epoch')
        plt.xlabel('Epoch')
        plt.ylabel('Score')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig('training_history.png')
        plt.close()
        
    except Exception as e:
        print(f"Error during training: {e}")
        raise e
        
if __name__ == "__main__":
    main()


=== GPU Setup ===
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU device count: 1
GPU device name: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Current GPU device: 0

Using device: cuda:0
Initial GPU Memory Allocated: 0.00 MB
Initial GPU Memory Reserved: 0.00 MB

Dataset Statistics:
Training images: 11607
Validation images: 2606
Test images: 2866
Classes: ['manipulated', 'original']



Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to C:\Users\HP/.cache\torch\hub\checkpoints\vgg16-397923af.pth
100%|██████████| 528M/528M [02:12<00:00, 4.16MB/s] 



Model Device Check:
Model device before: cpu
Model device after: cuda:0 


Epoch [1/20]
GPU Memory Allocated: 513.07 MB
GPU Memory Reserved: 518.00 MB
Batch [0/91] Loss: 0.7259 Acc: 50.78% GPU Memory: 1971.5MB
Batch [10/91] Loss: 0.6335 Acc: 61.36% GPU Memory: 1971.5MB
Batch [20/91] Loss: 0.5531 Acc: 67.67% GPU Memory: 1971.5MB
Batch [30/91] Loss: 0.4267 Acc: 71.04% GPU Memory: 1971.5MB
Batch [40/91] Loss: 0.3094 Acc: 73.72% GPU Memory: 1971.5MB
Batch [50/91] Loss: 0.3536 Acc: 75.51% GPU Memory: 1971.5MB
Batch [60/91] Loss: 0.2971 Acc: 77.33% GPU Memory: 1971.5MB
Batch [70/91] Loss: 0.1971 Acc: 78.97% GPU Memory: 1971.5MB
Batch [80/91] Loss: 0.2390 Acc: 80.44% GPU Memory: 1971.5MB
Batch [90/91] Loss: 0.2145 Acc: 81.60% GPU Memory: 1947.5MB

Epoch Summary:
Train Loss: 0.3968, Train Acc: 81.60%
Val Loss: 0.3056, Val Acc: 88.60%
Precision: 0.9132, Recall: 0.8520, F1: 0.8815
Learning Rate: 0.000105
Saved new best model with validation accuracy: 88.60%

Epoch [2/20]
GPU Memory Allocated: 1

In [4]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("PyTorch version:", torch.__version__)
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)

CUDA available: True
PyTorch version: 2.5.1+cu121
CUDA version: 12.1


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import VGG16_Weights
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix
import seaborn as sns
import sys

def test_model(model, test_loader):
    """Evaluate model on test set and display metrics"""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []

    criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # val_loss = running_loss / len(test_loader)
    # val_acc = accuracy_score(all_labels, all_preds) * 100
    # precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    
    # Calculate test metrics
    test_acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    # auc = roc_auc_score(all_labels, test_outputs_all)
    
    print("\n===== Test Set Evaluation =====")
    print(f"Accuracy: {test_acc:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    # print(f"AUC-ROC: {auc:.4f}")
    
    # Plot ROC curve
    # fpr, tpr, _ = roc_curve(all_labels, test_outputs_all)
    # plt.figure(figsize=(8, 6))
    # plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc:.4f})')
    # plt.plot([0, 1], [0, 1], 'k--')
    # plt.xlabel('False Positive Rate')
    # plt.ylabel('True Positive Rate')
    # plt.title('ROC Curve on Test Set')
    # plt.legend(loc='lower right')
    # plt.savefig('roc_curve.png')
    # plt.show()
    
    # Plot confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Real', 'Deepfake'], 
                yticklabels=['Real', 'Deepfake'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.savefig('confusion_matrix.png')
    plt.show()
    
    return {
        'accuracy': test_acc,
        'precision': precision, 
        'recall': recall,
        'f1': f1
    }

class DeepFakeDetector(nn.Module):
    def __init__(self, freeze_backbone=True):
        super(DeepFakeDetector, self).__init__()
        self.vgg16 = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        
        if freeze_backbone:
            for param in self.vgg16.features.parameters():
                param.requires_grad = False
        
        self.vgg16.classifier[6] = nn.Linear(4096, 2)
        
    def forward(self, x):
        return self.vgg16(x)

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

# Set device and ensure it's using CUDA
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device.type == 'cuda':
    print(f'Initial GPU Memory Allocated: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB')
    print(f'Initial GPU Memory Reserved: {torch.cuda.memory_reserved(0)/1024**2:.2f} MB')

# Dataset paths
data_dir = "../data/dataset"

# Hyperparameters
batch_size = 128  # Increased for GPU
num_epochs = 20
learning_rate = 0.001

# Data transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

try:
    train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
    val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
    test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=transform)
    
    print(f"\nDataset Statistics:")
    print(f"Training images: {len(train_dataset)}")
    print(f"Validation images: {len(val_dataset)}")
    print(f"Test images: {len(test_dataset)}")
    print(f"Classes: {train_dataset.classes}\n")
    
except Exception as e:
    print(f"Error loading datasets: {e}")
    print(f"Data directory: {os.path.abspath(data_dir)}")
    sys.exit(1)
    

# Create dataloaders with multiple workers for GPU
train_loader = DataLoader(train_dataset, 
                        batch_size=batch_size, 
                        shuffle=True,
                        num_workers=4,
                        pin_memory=True)
val_loader = DataLoader(val_dataset, 
                        batch_size=batch_size,
                        num_workers=4,
                        pin_memory=True)
test_loader = DataLoader(test_dataset, 
                        batch_size=batch_size,
                        num_workers=4,
                        pin_memory=True)

# Initialize model
model = DeepFakeDetector(freeze_backbone=True)
print("\nModel Device Check:")
print("Model device before:", next(model.parameters()).device)
model = model.to(device)
print("Model device after:", next(model.parameters()).device, "\n")

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = nn.DataParallel(model)

checkpoint = torch.load("best_model.pth")
model.load_state_dict(checkpoint['model_state_dict'])

print("Evaluating on test set...")
test_metrics = test_model(model, test_loader)

Using device: cuda:0
Initial GPU Memory Allocated: 0.00 MB
Initial GPU Memory Reserved: 0.00 MB

Dataset Statistics:
Training images: 11607
Validation images: 2606
Test images: 2866
Classes: ['manipulated', 'original']


Model Device Check:
Model device before: cpu
Model device after: cuda:0 



C:\Users\HP\AppData\Local\Temp\ipykernel_27324\640737612.py:173: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load("best_model.pth")


Evaluating on test set...

===== Test Set Evaluation =====
Accuracy: 0.9100
Precision: 0.9284
Recall: 0.8990
F1 Score: 0.9135


C:\Users\HP\AppData\Local\Temp\ipykernel_27324\640737612.py:79: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, datasets
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.models import VGG16_Weights
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_curve, auc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime
from tqdm import tqdm
import sys
import time
import json

def check_gpu():
    print("\n=== GPU Setup ===")
    print("PyTorch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("CUDA version:", torch.version.cuda)
        print("GPU device count:", torch.cuda.device_count())
        print("GPU device name:", torch.cuda.get_device_name(0))
        print("Current GPU device:", torch.cuda.current_device())
    print("================\n")

class DeepFakeDetector(nn.Module):
    def __init__(self, freeze_backbone=True):
        super(DeepFakeDetector, self).__init__()
        self.vgg16 = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1)
        
        if freeze_backbone:
            for param in self.vgg16.features.parameters():
                param.requires_grad = False
        
        self.vgg16.classifier[6] = nn.Linear(4096, 2)
        
    def forward(self, x):
        return self.vgg16(x)

def train_epoch(model, dataloader, criterion, optimizer, device, scheduler=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    # Create progress bar
    progress_bar = tqdm(dataloader, desc=f'Training',
                       file=sys.stdout, leave=True)
    
    for batch_idx, (inputs, labels) in enumerate(progress_bar):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        # Update progress bar
        accuracy = 100. * correct / total
        avg_loss = running_loss / (batch_idx + 1)
        progress_bar.set_postfix({
            'loss': f'{avg_loss:.4f}',
            'acc': f'{accuracy:.2f}%',
            'gpu_mem': f'{torch.cuda.memory_allocated()/1024**2:.1f}MB'
        })
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    
    return epoch_loss, epoch_acc, precision, recall, f1

def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []
    
    # Create progress bar
    progress_bar = tqdm(dataloader, desc=f'Validating',
                       file=sys.stdout, leave=True)
    
    with torch.no_grad():
        for inputs, labels in progress_bar:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_probs.extend(probs[:, 1].cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            # Update progress bar
            avg_loss = running_loss / (progress_bar.n + 1)
            accuracy = accuracy_score(all_labels, all_preds) * 100
            progress_bar.set_postfix({
                'loss': f'{avg_loss:.4f}',
                'acc': f'{accuracy:.2f}%'
            })
    
    val_loss = running_loss / len(dataloader)
    val_acc = accuracy_score(all_labels, all_preds) * 100
    precision, recall, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='binary')
    
    # Calculate ROC curve and AUC
    fpr, tpr, _ = roc_curve(all_labels, all_probs)
    roc_auc = auc(fpr, tpr)
    
    # Create confusion matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    return val_loss, val_acc, precision, recall, f1, roc_auc, cm, fpr, tpr

def plot_training_progress(history, output_dir):
    plt.figure(figsize=(15, 5))
    
    # Plot Loss
    plt.subplot(1, 3, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title('Loss vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Plot Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(history['train_acc'], label='Train Acc')
    plt.plot(history['val_acc'], label='Val Acc')
    plt.title('Accuracy vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    
    # Plot F1 Score
    plt.subplot(1, 3, 3)
    plt.plot(history['val_f1'], label='F1 Score')
    plt.plot(history['val_precision'], label='Precision')
    plt.plot(history['val_recall'], label='Recall')
    plt.title('Metrics vs Epoch')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'training_history.png'))
    plt.close()

def main():
    # Set for deterministic behavior
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        
    # Create output directory
    output_dir = "training_output"
    os.makedirs(output_dir, exist_ok=True)

    # Check GPU setup
    check_gpu()
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    if device.type == 'cuda':
        print(f'Initial GPU Memory: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB')
    
    # Dataset paths
    data_dir = "../data/dataset"
    
    # Hyperparameters
    batch_size = 32  # Reduced batch size for better stability
    num_epochs = 20
    learning_rate = 0.001
    
    # Data transforms with normalization
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    # Load datasets
    try:
        train_dataset = datasets.ImageFolder(os.path.join(data_dir, 'train'), transform=transform)
        val_dataset = datasets.ImageFolder(os.path.join(data_dir, 'val'), transform=transform)
        test_dataset = datasets.ImageFolder(os.path.join(data_dir, 'test'), transform=transform)
        
        print(f"\nDataset Statistics:")
        print(f"Training images: {len(train_dataset)}")
        print(f"Validation images: {len(val_dataset)}")
        print(f"Test images: {len(test_dataset)}")
        print(f"Classes: {train_dataset.classes}\n")
        
    except Exception as e:
        print(f"Error loading datasets: {e}")
        return
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                            shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                          num_workers=4, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                           num_workers=4, pin_memory=True)
    
    # Initialize model
    model = DeepFakeDetector(freeze_backbone=True)
    model = model.to(device)
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)
    
    # Training history
    history = {
        'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
        'val_precision': [], 'val_recall': [], 'val_f1': [], 'val_auc': []
    }
    
    best_val_acc = 0.0
    start_time = time.time()
    
    try:
        print("\nStarting training...\n")
        
        for epoch in range(num_epochs):
            epoch_start_time = time.time()
            print(f"\nEpoch [{epoch+1}/{num_epochs}]")
            
            if device.type == 'cuda':
                print(f'GPU Memory: {torch.cuda.memory_allocated(0)/1024**2:.2f} MB')
            
            # Train
            train_loss, train_acc, train_precision, train_recall, train_f1 = train_epoch(
                model, train_loader, criterion, optimizer, device)
            
            # Validate
            val_loss, val_acc, val_precision, val_recall, val_f1, val_auc, cm, fpr, tpr = validate(
                model, val_loader, criterion, device)
            
            # Update learning rate
            scheduler.step(val_loss)
            current_lr = optimizer.param_groups[0]['lr']
            
            # Update history
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)
            history['val_precision'].append(val_precision)
            history['val_recall'].append(val_recall)
            history['val_f1'].append(val_f1)
            history['val_auc'].append(val_auc)
            
            # Calculate epoch time
            epoch_time = time.time() - epoch_start_time
            
            # Print epoch summary
            print(f"\nEpoch Summary:")
            print(f"Time: {epoch_time:.2f}s")
            print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
            print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
            print(f"Val F1: {val_f1:.4f}, Val AUC: {val_auc:.4f}")
            print(f"Learning Rate: {current_lr:.2e}")
            
            # Save best model
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'val_acc': val_acc,
                    'val_f1': val_f1,
                    'val_auc': val_auc
                }, os.path.join(output_dir, 'best_model.pth'))
                print(f"Saved new best model with validation accuracy: {val_acc:.2f}%")
            
            # Plot current progress
            plot_training_progress(history, output_dir)
            
            # Memory cleanup
            torch.cuda.empty_cache()
        
        total_time = time.time() - start_time
        print(f"\nTraining completed in {total_time/60:.2f} minutes!")
        
        # Save final history
        with open(os.path.join(output_dir, 'training_history.json'), 'w') as f:
            json.dump(history, f, indent=4)
        
    except Exception as e:
        print(f"\nError during training: {e}")

if __name__ == "__main__":
    main()


=== GPU Setup ===
PyTorch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU device count: 1
GPU device name: NVIDIA GeForce RTX 3050 Ti Laptop GPU
Current GPU device: 0

Using device: cuda:0
Initial GPU Memory: 0.00 MB

Dataset Statistics:
Training images: 11607
Validation images: 2606
Test images: 2866
Classes: ['manipulated', 'original']


Starting training...


Epoch [1/20]
GPU Memory: 513.07 MB
Training:   8%|▊         | 30/363 [00:24<04:34,  1.21it/s, loss=1.1066, acc=64.69%, gpu_mem=1915.9MB] 


KeyboardInterrupt: 